# Streaming Assignment Evaluation

This notebook presents a representative student submission for the Streaming Assignment.
Many students turned in work similar to this example. A single procedural script that decodes a Morse binary file and prints the result.

This submission shows effort and demonstrates that the student worked through the notebooks.
However, it does **not** meet the actual assignment requirements.

In the sections that follow, we will:

- Examine the submitted code
- Identify where it aligns with notebook patterns
- Document where it fails to meet the specification
- Explain what the assignment actually required
- Provide clear guidance for improvement

The goal is not to criticize the student, but to help everyone understand the difference between *using notebook patterns* and *adapting notebook patterns into a full streaming pipeline with modular class design*. The course is almost half done, so we need to understand how to complete an assignment successfully.


In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import csv # For reading the morse code table

# The BitIterator class iterates each byte, bit by bit, into strings from the binary data.
class BitIterator:
    def __init__(self, data):
        self.data = data # Binary data
        self.bytes = 0 # First byte
        self.bits = 0 # First bit

    def __iter__(self):
        return self # Iterator object

    def __next__(self):
        # Stops the loop if the current byte index is at least the total number of bytes.
        if self.bytes >= len(self.data):
            raise StopIteration

        # Section: shift_right() from Bits&Bytes
        # - Masks the current byte.
        # - Extracts bit by bit of the current byte.
        byte = self.data[self.bytes]
        bit = (byte >> (7 - self.bits)) & 1

        self.bits += 1

        if self.bits == 8:
            self.bits = 0
            self.bytes += 1

        return str(bit)

# The decode() function converts 2-bit chunks into morse code sequences.
def decode(bit_iterator, table):
    chunk = ""
    sequence = ""

    # Creates the morse code sequence bit by bit.
    for bit in bit_iterator:
        chunk += bit

        # Checks for exactly 2 bits in the chunk to continue the process
        if len(chunk) != 2:
            continue

        if chunk == "10": # DOT = 10 = .
            sequence += "."
        elif chunk == "01": # DASH = 01 = -
            sequence += "-"
        elif chunk == "00": # End of letter = 00
            if sequence:
                yield table.get(sequence, "?")
                sequence = ""
        elif chunk == "11": # End of word = 11
            if sequence:
                yield table.get(sequence, "?")
                sequence = ""
            yield " "

        chunk = ""

    # Ends the sequence in case it is still waiting to be decoded.
    if sequence:
        yield table.get(sequence, "?")

# The morse_dict() function returns a dictionary of all morse codes and characters based on the codes.
def morse_dict(file):
    dictionary = {}

    with open(file, "r", newline="", encoding="utf-8") as f:
        table = csv.reader(f) # Reads CSV file.

        for row in table: # Each row in the table will produce a character related to its hex value and morse code.
            hex_value = row[0].strip() # Stripped hex value.
            morse_code = row[1].strip() # Stripped morse code.
            dictionary[morse_code] = chr(int(hex_value, 16)) # Puts the morse code with its character in the dictionary.

    return dictionary

# Section: StreamDemo from BinaryPipes
# Used in: MainProgram
# - Since we are decoding a singular binary file, the class is centered around binary streaming.
# - Two file buttons are added for us to select both binary and CSV files.
# - A decode button is added to produce the output text of the decoded file.
class MainProgram:
    def __init__(self, root):
        self.root = root
        root.title("Morse Code Bitstream")
        root.geometry("1200x800")

        title = tk.Label(root, text="Morse Code Bitstream", font=("Arial", 28, "bold"))
        title.pack(pady=20, anchor="w")

        control_frame = tk.Frame(root)
        control_frame.pack(pady=10, anchor="w", fill="x")

        tk.Button(control_frame, text="Choose Binary File", font=("Arial", 16),
                  command=self.choose_binary_file).grid(row=0, column=0, padx=10, pady=5, sticky="w")

        self.binary_label = tk.Label(control_frame, text="No binary file selected",
                                     width=60, anchor="w", font=("Arial", 14))
        self.binary_label.grid(row=0, column=1, padx=10, sticky="w")

        tk.Button(control_frame, text="Choose CSV File", font=("Arial", 16),
                  command=self.choose_csv_file).grid(row=1, column=0, padx=10, pady=5, sticky="w")

        self.csv_label = tk.Label(control_frame, text="No CSV file selected",
                                  width=60, anchor="w", font=("Arial", 14))
        self.csv_label.grid(row=1, column=1, padx=10, sticky="w")

        tk.Button(control_frame, text="Decode", font=("Arial", 16),
                  command=self.decode_file).grid(row=2, column=0, padx=10, pady=5, sticky="w")

        self.output = tk.Text(root, font=("Consolas", 16))
        self.output.pack(fill="both", expand=True, padx=10, pady=10)

        self.binary_filename = None
        self.csv_filename = None

    # Section: choose_file() from BinaryPipes
    # Used in: choose_binary_file()
    # - Asks for only Binary file types.
    def choose_binary_file(self):
        self.binary_filename = filedialog.askopenfilename(filetypes=[("Binary files", "*.bin")])
        if self.binary_filename:
            self.binary_label.config(text=self.binary_filename)

    # Section: choose_file() from BinaryPipes
    # Used in: choose_csv_file()
    # - Asks for only CSV file types.
    def choose_csv_file(self):
        self.csv_filename = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv")])
        if self.csv_filename:
            self.csv_label.config(text=self.csv_filename)

    # The decode_file() class function sets the decoding process to the binary file.
    # Outputs the decoded text.
    def decode_file(self):
        # Section: load_stream() from BinaryPipes
        # - Unchosen files for either will not return or output anything.
        if not self.binary_filename or not self.csv_filename:
            return

        # Section: stream_binary() from BinaryPipes
        # - Opens a file in binary mode
        # - Combines all the decoded characters into a single string
        # - Inserts the string into the output text box
        try:
            with open(self.binary_filename, "rb") as bf:
                data = bf.read()
            bits = BitIterator(data)
            morse_table = morse_dict(self.csv_filename)
            decoded_text = "".join(decode(bits, morse_table))
            self.output.insert(tk.END, decoded_text)

        # If a decode error is present, a message box will show the type of error.
        except Exception as error:
            messagebox.showerror("Decode Error", str(error))

if __name__ == "__main__":
    root = tk.Tk()
    MainProgram(root)
    root.mainloop()



# Where This Submission Meets Expectations

Before we discuss the missing requirements, it is important to acknowledge what this submission *does* accomplish. Many students produced work similar to this, and these strengths show genuine engagement with the course notebooks.

## 1. Correct Bit-Level Iteration
The `BitIterator` class demonstrates a clear understanding of the bit‑extraction techniques from **Bits & Bytes**:

- Uses shifting and masking correctly
- Iterates through each byte and each bit
- Produces `"0"` and `"1"` strings for the decoder

This shows the student understood the notebook’s bit‑level operations.

## 2. Correct Morse Decoding Logic
The `decode()` generator:

- Accumulates bits into 2‑bit chunks
- Converts `"10"` to DOT and `"01"` to DASH
- Handles `"00"` (end of letter) and `"11"` (end of word)
- Yields decoded characters
- Uses a fallback `"?"` for unknown sequences

This matches the expected Morse decoding behavior and shows understanding of generator patterns.

## 3. Correct CSV Table Loading
The `morse_dict()` function:

- Reads a CSV file
- Extracts hex values
- Converts them to characters
- Builds a dictionary mapping Morse sequences to ASCII

This is a faithful adaptation of the CSV‑reading patterns from earlier notebook exercises.

## 4. Working Tkinter GUI
The submission includes a functional Tkinter interface:

- File selection buttons
- Labels showing chosen files
- A “Decode” button
- A text box for output
- Error handling via `messagebox.showerror`

This demonstrates effort to integrate Tkinter into the assignment, which many students did not attempt.

## 5. Clear Effort to Reference Notebook Sections
The comments explicitly reference:

- Bits & Bytes
- BinaryPipes
- StreamDemo
- shift_right
- stream_binary
- choose_file

This shows the student tried to connect notebook content to the assignment, even if the adaptation was incomplete.

## 6. The Program Works
Given a valid `.bin` file and a valid `.csv` Morse table:

- The program runs
- The bits are streamed
- The Morse is decoded
- The output appears in the GUI

Functionally, this is a working Morse decoder.

---

Overall, this submission demonstrates **real effort**, **real understanding of the notebooks**, and **a working implementation** of Morse decoding. These strengths matter, and they show that the student is learning the core concepts.

The next section will explain where the submission falls short of the *full* assignment specification.
```


## Where This Submission Fails the Specification

Although this submission shows real effort and demonstrates understanding of several notebook concepts, it does **not** meet the Streaming Assignment requirements.
The issues below are structural and conceptual gaps that prevent the submission from satisfying the specification.

---

## 1. Missing Required Multi‑Class Architecture

The assignment required a **multi‑stage, class‑based streaming pipeline**, modeled after:

- CaesarCipher (class layout, private state, properties)
- Bits & Bytes (bit extraction logic)
- BinaryPipes (streaming pipeline)
- Tkinter showcase (GUI class)

This submission includes only one meaningful class (`MainProgram`) and one utility class (`BitIterator`).
It does **not** implement:

- a Decoder class
- a Stream class
- a Display class
- a pipeline that connects them

Decoding is procedural and happens inside a single method.

---

## 2. No Streaming Pipeline

The assignment required a **true streaming pipeline**, where:

1. A binary stream feeds bits
2. Bits feed a decoder
3. Decoder feeds a character stream
4. Character stream feeds the GUI

This submission performs decoding in one step:

```python
decoded_text = "".join(decode(bits, morse_table))
```

There is no pipeline, no chaining, and no modular flow — just a single decode call.

---

## 3. No Adaptation of Notebook Patterns

The assignment required students to **adapt** notebook patterns, not copy them.

This submission:

- copies BitIterator almost directly
- copies decode logic almost directly
- copies CSV loading almost directly
- copies BinaryPipes file‑selection patterns almost directly

There is no meaningful adaptation, only reuse.
Comments reference notebook sections, but the code does not demonstrate actual integration of those patterns into a new architecture.

---

## 4. GUI Does Not Represent a Streaming Display

The Tkinter GUI is present, but it does not meet the assignment’s intent:

- It displays the **final decoded text**, not a stream
- It does not show incremental output
- It does not demonstrate a pipeline feeding the GUI
- It does not use a scrolled text widget in a streaming manner

The GUI is essentially a file picker plus a text box.

---

## 5. No Private State or Properties

The assignment required:

- private attributes (`__state`, `__position`, etc.)
- properties (`@property`)
- class encapsulation

This submission uses public attributes everywhere:

```python
self.bytes
self.bits
self.binary_filename
self.csv_filename
```

There are no properties, no encapsulation, and no class‑level state management.

---

## 6. No Error‑Resilient Streaming

The assignment required:

- handling malformed Morse sequences
- handling incomplete bit pairs
- handling unexpected end‑of‑file
- fallback behavior beyond `"?"`

This submission:

- assumes perfect input
- does not validate bit pairs
- does not guard against partial bytes
- does not handle corrupted streams

The decode logic will silently produce incorrect output if the stream is malformed.

---

## 7. No Documentation of Design Decisions

The assignment required:

- explanation of how notebook patterns were adapted
- justification of architectural choices
- demonstration of conceptual understanding

This submission includes comments, but they are:

- descriptive rather than explanatory
- referencing notebooks without showing adaptation
- not documenting design reasoning

The comments do not demonstrate the required level of reflection.

---

## 8. Procedural Instead of Modular

The assignment required a modular design with:

- separate classes
- separate responsibilities
- a pipeline architecture

This submission is fundamentally **procedural**:

- decode logic is a free function
- CSV loading is a free function
- streaming is done in one line
- GUI logic is tightly coupled to decoding logic

This violates the core architectural requirement of the assignment.

---

# Summary

This submission **works**, and it shows real effort.
But it does **not** meet the specification because it lacks:

- the required multi‑class architecture
- a streaming pipeline
- notebook pattern adaptation
- modular design
- robust error handling
- documented design decisions

These are normal first‑attempt mistakes, but they are also the reason this submission cannot receive full credit under the assignment rubric.



## What the Assignment Actually Required

To understand why the representative submission does not meet the specification, we need to restate the actual requirements of the Streaming Assignment.
The goal of the assignment was not simply to decode Morse code.
The goal was to **demonstrate mastery of the course’s modular, class‑based, streaming architecture** by adapting patterns from multiple notebooks into a unified system.

Below is a clear summary of what the assignment required.

---

## 1. A Multi‑Class, Modular Architecture

The assignment required **multiple classes**, each with a clear responsibility:

- **BitIterator** — bit‑level access (Bits & Bytes)
- **Decoder** — converts bit pairs into characters (Generator exercises)
- **Stream** — manages binary input (BinaryPipes)
- **Display** — GUI output (Tkinter showcase)
- **MainProgram** — coordinates the pipeline

Each class needed:

- private state (`__attribute`)
- properties (`@property`)
- a meaningful `__repr__`
- separation of responsibilities

The representative submission uses only one meaningful class (`MainProgram`) and does not implement the required architecture.

---

## 2. A True Streaming Pipeline

The assignment required a **pipeline**, not a single decode call.

A correct pipeline looks like this:

```
BinaryStream  →  BitIterator  →  Decoder  →  CharacterStream  →  GUI Display
```

Each stage feeds the next.

The representative submission does everything in one step:

```
decoded_text = "".join(decode(bits, morse_table))
```

This is not a pipeline.

---

## 3. Adaptation of Notebook Patterns

Students were expected to **adapt** patterns from:

- CaesarCipher (class layout, private state)
- Bits & Bytes (bit extraction)
- BinaryPipes (streaming)
- Generator exercises (yielding characters)
- Tkinter textbox tab (GUI output)

Adaptation means:

- changing structure
- integrating patterns
- combining ideas
- building something new

Copying notebook code directly was not sufficient.

---

## 4. A Streaming GUI Display

The Tkinter window needed to:

- show decoded output **as it streams**
- use a scrolled text widget
- demonstrate incremental updates
- reflect the pipeline’s output

The representative submission displays only the **final** decoded text, not a stream.

---

## 5. Robust, Error‑Resilient Streaming

The assignment required:

- handling malformed Morse sequences
- handling incomplete bit pairs
- handling unexpected end‑of‑file
- fallback behavior beyond `"?"`
- safe termination of streams

Students needed to show they understood how real streaming systems behave under imperfect conditions.

---

## 6. Documented Design Decisions

The assignment required:

- explanations of how notebook patterns were adapted
- justification of architectural choices
- comments that demonstrate conceptual understanding

The representative submission includes comments, but they are descriptive rather than reflective.

---

## Summary

The Streaming Assignment was not a Morse decoder exercise.
It was a **modular architecture exercise**, a **pipeline exercise**, and a **notebook‑adaptation exercise**.

A correct submission needed:

- multiple classes
- private state and properties
- a streaming pipeline
- a streaming GUI
- robust error handling
- documented design decisions

Understanding these requirements clarifies why many submissions — including the representative example — did not meet the specification.
```



## Correct Architecture Diagram

To clarify what the Streaming Assignment expected, this section presents the
**correct modular architecture** in a visual, pipeline‑style diagram.
This is the structure students were supposed to build by *adapting* patterns from
CaesarCipher, Bits & Bytes, BinaryPipes, and the Tkinter showcase.

The key idea:
**Each stage is a class. Each class has one job. Each stage streams data to the next.**

---

## High‑Level Pipeline

```
 ┌────────────────┐    ┌────────────────┐    ┌────────────────┐    ┌──────────────────┐    ┌────────────────────┐
 │  BinaryStream  │ -> │  BitIterator   │ -> │    Decoder     │ -> │ CharacterStream  │ -> │   TkinterDisplay   │
 └────────────────┘    └────────────────┘    └────────────────┘    └──────────────────┘    └────────────────────┘
```

Each box is a class.
Each arrow represents a **stream of data** flowing from one stage to the next.

---

## Stage‑by‑Stage Breakdown

### 1. **BinaryStream (BinaryPipes adaptation)**
- Opens the `.bin` file
- Streams raw bytes
- Provides a generator or iterator interface
- Private state: `__filename`, `__file`, `__buffer`
- Property: `@property filename`

### 2. **BitIterator (Bits & Bytes adaptation)**
- Receives bytes from `BinaryStream`
- Extracts bits using shifting and masking
- Yields `"0"` and `"1"` strings
- Private state: `__data`, `__position`
- Property: `@property total_bits`

### 3. **Decoder (Generator exercises adaptation)**
- Receives bits from `BitIterator`
- Groups bits into 2‑bit chunks
- Builds Morse sequences
- Yields decoded characters
- Private state: `__sequence`, `__chunk`
- Property: `@property last_sequence`

### 4. **CharacterStream**
- Receives characters from `Decoder`
- Buffers them or streams them directly
- Provides incremental output
- Private state: `__buffer`
- Property: `@property text`

### 5. **TkinterDisplay (Tkinter showcase adaptation)**
- Receives characters from `CharacterStream`
- Updates the GUI incrementally
- Uses a scrolled text widget
- Private state: `__textbox`
- Property: `@property contents`

---

## How the Pieces Fit Together

The assignment required students to **connect** these classes:

```
stream = BinaryStream("Morse.bin")
bits   = BitIterator(stream)
chars  = Decoder(bits)
gui    = TkinterDisplay()

for ch in chars:
    gui.write(ch)
```

This is the essence of a streaming pipeline:

- **Data flows forward**
- **Classes stay independent**
- **Each class has one responsibility**
- **The GUI updates as the stream produces output**

---

## Why This Architecture is important

This design demonstrates:

- modularity
- private state
- properties
- generator pipelines
- Tkinter integration
- adaptation of notebook patterns
- real streaming behavior

It is the architecture the assignment was designed to teach.

---

This diagram sets the stage for the next section, where we will compare this architecture to the representative student submission and highlight the structural differences.

## Improved Version of the Code (Excerpts Only)

This section presents **selected excerpts** from a properly structured solution.
These are *not* full implementations. They are architectural examples showing how the assignment should have been organized.

The goal is to demonstrate:

- private state
- properties
- class‑based modularity
- a true streaming pipeline
- separation of responsibilities
- adaptation of notebook patterns

These excerpts are intentionally incomplete so students can learn from the structure without receiving a full solution.

---

## 1. BitIterator (Improved)

```python
class BitIterator:
    def __init__(self, data):
        self.__data = data
        self.__position = 0

    @property
    def total_bits(self):
        return len(self.__data) * 8

    def __iter__(self):
        return self

    def __next__(self):
        if self.__position >= self.total_bits:
            raise StopIteration

        byte_index = self.__position // 8
        bit_index  = 7 - (self.__position % 8)

        byte = self.__data[byte_index]
        bit  = (byte >> bit_index) & 1

        self.__position += 1
        return "1" if bit else "0"
```

**Key improvements:**

- private state (`__data`, `__position`)
- property (`total_bits`)
- cleaner bit‑extraction logic
- matches the CaesarCipher class pattern

---

## 2. Decoder Class (New)

```python
class Decoder:
    def __init__(self, table):
        self.__table = table
        self.__sequence = ""
        self.__chunk = ""

    def __iter__(self):
        return self

    def feed(self, bit):
        self.__chunk += bit
        if len(self.__chunk) < 2:
            return None

        pair = self.__chunk
        self.__chunk = ""

        if pair == "10":
            self.__sequence += "."
        elif pair == "01":
            self.__sequence += "-"
        elif pair == "00":
            char = self.__table.get(self.__sequence, "?")
            self.__sequence = ""
            return char
        elif pair == "11":
            char = self.__table.get(self.__sequence, "?")
            self.__sequence = ""
            return char + " "
```

**Key improvements:**

- decoder is now a **class**, not a free function
- private state (`__sequence`, `__chunk`)
- incremental decoding
- matches generator‑pipeline patterns

---

## 3. CharacterStream (New)

```python
class CharacterStream:
    def __init__(self):
        self.__buffer = []

    def push(self, ch):
        if ch is not None:
            self.__buffer.append(ch)

    @property
    def text(self):
        return "".join(self.__buffer)
```

**Key improvements:**

- separates character buffering from decoding
- provides a clean interface for the GUI
- demonstrates modularity

---

## 4. TkinterDisplay (Improved)

```python
class TkinterDisplay:
    def __init__(self, root):
        self.__box = tk.Text(root, font=("Consolas", 16))
        self.__box.pack(fill="both", expand=True)

    def write(self, ch):
        self.__box.insert(tk.END, ch)
        self.__box.see(tk.END)
```

**Key improvements:**

- incremental streaming output
- GUI updates as characters arrive
- matches the Tkinter notebook pattern

---

## 5. MainProgram (Pipeline Assembly)

```python
stream = BinaryStream(filename)
bits   = BitIterator(stream)
decode = Decoder(morse_table)
chars  = CharacterStream()
gui    = TkinterDisplay(root)

for bit in bits:
    ch = decode.feed(bit)
    chars.push(ch)
    gui.write(ch)
```

**Key improvements:**

- true streaming pipeline
- each class has one responsibility
- GUI updates in real time
- demonstrates adaptation of notebook patterns

---

These excerpts illustrate the architecture the assignment required:

- multiple classes
- private state
- properties
- modular design
- streaming pipeline
- incremental GUI output

They are **not** a full solution, but they show the structure students were expected to build.

## Wrapup

This notebook has shown a representative Streaming Assignment submission, examined
its strengths, identified where it fell short of the specification, and clarified
what the assignment actually required.

The goal of this discussion is not to criticize anyone.
It is to help you understand the difference between:

- **copying notebook patterns**, and
- **adapting notebook patterns into a full, modular, streaming architecture.**

Many of you demonstrated real effort and real understanding of the notebooks.
But the Streaming Assignment was designed to push you beyond procedural scripts
and into **multi‑class design**, **private state**, **properties**, **pipelines**, and
**incremental GUI streaming**.

Here are the key takeaways:

---

## What you did well
- You understood bit extraction.
- You understood Morse decoding.
- You understood CSV table loading.
- Many of you built a working Tkinter interface.
- You showed real engagement with the course notebooks.

These are meaningful accomplishments.

---

## What you need to improve
- Build **multiple classes**, each with one responsibility.
- Use **private state** and **@property** to manage internal data.
- Construct a **streaming pipeline**, not a single decode call.
- Adapt notebook patterns — don’t just reuse them.
- Stream output **incrementally** into the GUI.
- Document your **design decisions**, not just your code.

These skills matter because they are the foundation of real software engineering.

---

## Why this is important
The Streaming Assignment was not about Morse code.
It was about:

- modular architecture
- class design
- streaming systems
- pipeline thinking
- adapting patterns
- building maintainable software

These are the skills that separate “working code” from “engineered code.”

---

## Moving forward
If your submission looked like the representative example, you are **not behind**.
You are learning exactly what you need to learn, and this lecture is meant to help
you take the next step.

Use this notebook as a guide:

- Study the architecture diagram
- Review the improved code excerpts
- Compare them to your submission
- Practice building pipelines in small steps

You will get better at this quickly. Modular design becomes natural with practice.
